# 1.0. Define the task and split the dataset

Identify your classes and divide your images into training, validation, and test sets. Keep the test set separate for final evaluation; use validation results to guide training choices.

## 1.1 IMPORTS — Code: imports for PyTorch, ImageFolder, and dataset splitting.

In [3]:
# 1.1.
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch import torch.Generator
from torch import torch.utils.data.random_split

SyntaxError: invalid syntax (861673666.py, line 4)

## 1.2 SET DATASET PATH AND CLASSES — Markdown + code: set the image-folder path and inspect class names and image counts.

ImageFolder expects one subfolder per category, with images inside each class folder. Set data_dir to the parent folder, then inspect the detected classes and image counts.

In [ ]:
from collections import Counter
from pathlib import Path
from torchvision.datasets import ImageFolder

data_dir = Path("path/to/images")  # Replace with your dataset folder

if not data_dir.is_dir():
    raise FileNotFoundError(f"Dataset folder not found: {data_dir}")

dataset = ImageFolder(root=data_dir)

class_counts = Counter(dataset.targets)

print(f"Total images: {len(dataset)}")
print(f"Number of classes: {len(dataset.classes)}")

for class_index, class_name in enumerate(dataset.classes):
    print(f"{class_name}: {class_counts[class_index]} images")


# 2.0. Choose a pretrained model and matched image pre-processing

Choose a model and use the image size, normalization, and other preprocessing expected by its pretrained weights.

## 2.1 IMPORTS — Code: imports for torchvision.models, transforms, and DataLoader.

In [ ]:
# 2.1. 
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

## 2.2 SELECT PRETRAINED MODEL — Markdown + code: choose a model and its pretrained weights.

This example selects a ResNet-18 pretrained on ImageNet. Keep weights available for the preprocessing step and use the same model architecture when creating your working model in Step 3.

In [ ]:
from torchvision import models

model_name = "resnet18"
weights = models.ResNet18_Weights.DEFAULT

# Load the pretrained model and weights.
pretrained_model = models.resnet18(weights=weights)

print(f"Selected model: {model_name}")
print(f"Pretrained weights: {weights}")
print(f"Pretraining categories: {len(weights.meta['categories'])}")


## 2.3 SET PREPROCESSING AND LOADERS — Markdown + code: apply the weights’ expected preprocessing and create data loaders.

Use the preprocessing associated with the selected pretrained weights. This example applies the same preprocessing to all splits, then creates reproducible splits and data loaders. For stronger training augmentation, you can add separate training transforms later.

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder

preprocess = weights.transforms()
dataset = ImageFolder(root=data_dir, transform=preprocess)

train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)
train_set, val_set, test_set = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator,
)

batch_size = 32

train_loader = DataLoader(
    train_set, batch_size=batch_size, shuffle=True
)
val_loader = DataLoader(
    val_set, batch_size=batch_size, shuffle=False
)
test_loader = DataLoader(
    test_set, batch_size=batch_size, shuffle=False
)

print(f"Train: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")


# 3.0 Create a separate working model and replace its classifier for selected categories

Load the pretrained weights into a working model, replace its final classification layer to match your classes, and keep the original checkpoint untouched. Save your adapted model to a different checkpoint path.

## 3.1 IMPORTS — Code: imports for PyTorch and torch.nn.

In [4]:
# 3.1. IMPORTS
import copy
import torch
from torch import nn

## 3.2 CREATE A SEPARATE MODEL — Markdown + code: load pretrained weights into a new model instance.

Create a new ResNet-18 instance using the selected pretrained weights. This gives you a separate model to adapt and train; the pretrained_model from Step 2 remains unchanged.

In [5]:
from torchvision import models

model = models.resnet18(weights=weights)

print(f"Working model created: {model_name}")
print(f"Original pretrained model still available: {pretrained_model is not model}")


NameError: name 'weights' is not defined

## 3.3 REPLACE THE CLASSIFIER — Markdown + code: set the classifier’s output count to the number of categories.

For ResNet-18, the classifier is model.fc. Replace it with a layer whose output count equals the number of categories. This assumes model is your separate working model and dataset.classes contains the category names.

In [ ]:
import torch.nn as nn

num_classes = len(dataset.classes)
in_features = model.fc.in_features

model.fc = nn.Linear(in_features, num_classes)

print(f"Categories ({num_classes}): {dataset.classes}")
print(model.fc)


## 3.4 SAVE AN INITIAL CHECKPOINT — Markdown + code: save to a new path, not over the original pretrained checkpoint.

Save the adapted model’s initial weights to a new file path. This preserves the original pretrained checkpoint. For ResNet-18, the final layer has already been replaced as in Step 3.3.

In [ ]:
from pathlib import Path
import torch

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

initial_checkpoint_path = checkpoint_dir / "resnet18_adapted_initial.pth"

torch.save(
    {
        "model_name": model_name,
        "class_names": dataset.classes,
        "model_state_dict": model.state_dict(),
    },
    initial_checkpoint_path,
)

print(f"Saved adapted model checkpoint to: {initial_checkpoint_path}")


# 4.0. Freeze the pretrained backbone and train the new classifier head

Freeze the pretrained backbone and train the new classification head. This feature-extraction approach is a useful baseline, particularly for smaller datasets.

## 4.1 IMPORTS — Code: imports for the loss function and optimizer.

In [ ]:
# 4.1. 
import torch
from torch import nn, optim

## 4.2 FREEZE BACKBONE AND CONFIGURE TRAINING — Markdown + code: freeze pretrained layers, select device, loss, and optimizer.

This ResNet-18 example freezes all pretrained layers except the replacement classifier. It selects CUDA when available and otherwise uses the CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Freeze all pretrained parameters.
for parameter in model.parameters():
    parameter.requires_grad = False

# Train only the replacement classifier.
for parameter in model.fc.parameters():
    parameter.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

print(f"Training device: {device}")


## 4.3 TRAIN CLASSIFIER HEAD — Markdown + code: run the training loop for the new classifier.

This example trains the new classifier head for several epochs. It assumes train_loader, model, optimizer, criterion, and device are defined in earlier cells.

In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    total_examples = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total_examples += batch_size

    average_loss = running_loss / total_examples
    print(f"Epoch {epoch + 1}/{epochs} — training loss: {average_loss:.4f}")


# 5.0. Evaluate the validataion dataset after each epoch and save best checkpoint

Track training and validation metrics after each epoch. Save a separate checkpoint whenever validation performance improves.

## 4.3 TRAIN CLASSIFIER HEAD — Markdown + code: run the training loop for the new classifier.

In [ ]:
# 5.1. 
import torch
from sklearn.metrics import accuracy_score

## 5.2 EVALUATE ON VALIDATION SET — Markdown + code: calculate validation loss and accuracy after each epoch.

Define this function once, then call it after each training epoch. It calculates average validation loss and accuracy without updating model weights.

In [ ]:
def evaluate_on_validation(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_examples = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            predictions = outputs.argmax(dim=1)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            correct += (predictions == labels).sum().item()
            total_examples += batch_size

    average_loss = total_loss / total_examples
    accuracy = correct / total_examples
    return average_loss, accuracy


# Call this after each training epoch:
val_loss, val_accuracy = evaluate_on_validation(
    model, val_loader, criterion, device
)
print(f"Validation loss: {val_loss:.4f} | Validation accuracy: {val_accuracy:.4f}")


## 5.3 TRACK HISTORY AND SAVE BEST CHECKPOINT — Markdown + code: record epoch metrics and save weights when validation performance improves.

Add this inside the epoch loop, after calculating train_loss, val_loss, and val_accuracy. It records metrics for every epoch and saves model weights when validation accuracy improves.

In [ ]:
from pathlib import Path
import torch

history = []
best_val_acc = 0.0
checkpoint_path = Path("checkpoints/best_model.pth")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

# Inside the epoch loop, after validation:
history.append({
    "epoch": epoch + 1,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "val_accuracy": val_accuracy,
})

if val_accuracy > best_val_acc:
    best_val_acc = val_accuracy
    torch.save(
        {
            "epoch": epoch + 1,
            "val_accuracy": val_accuracy,
            "model_state_dict": model.state_dict(),
        },
        checkpoint_path,
    )
    print(f"Saved best checkpoint: {checkpoint_path}")


# 6.0. Fine-tune selected pre-trained layers if classfier-only training plateaus

If classifier-only performance plateaus, unfreeze some pretrained layers—often starting with later layers—and continue training. Use a lower learning rate for pretrained layers than for the new classifier head.

In [ ]:
## 6.1. IMPORTS 

In [ ]:
# 6.1. 
import torch
from torch import optim

## 6.2. CHOOSE LAYERS TO UNFREEZE — Markdown: explain which later layers to unfreeze and why.

For ResNet-18, start with layer4, the final pretrained feature block, plus the new classifier. If validation performance still plateaus, try unfreezing layer3 as well. Unfreezing fewer layers first limits how much of the pretrained representation changes. This cell records the choice; Step 6.3 applies it.

In [ ]:
# ResNet-18 layer names, ordered from later to earlier feature blocks.
layers_to_unfreeze = ["layer4"]

# If needed in a later experiment, try:
# layers_to_unfreeze = ["layer3", "layer4"]

print("Pretrained layers selected for fine-tuning:", layers_to_unfreeze)
print("New classifier selected for training: fc")


## 6.3. CONFIGURE FINE-TUNING — Code: unfreeze selected layers and set lower learning rates for pretrained layers.

For ResNet-18, this example unfreezes `layer4` and the classifier. It assigns the pretrained layer a lower learning rate than the newly initialized classifier.

In [ ]:
import torch.optim as optim

# Freeze everything first.
for parameter in model.parameters():
    parameter.requires_grad = False

# Unfreeze the final pretrained block and the classifier.
for parameter in model.layer4.parameters():
    parameter.requires_grad = True

for parameter in model.fc.parameters():
    parameter.requires_grad = True

optimizer = optim.Adam(
    [
        {"params": model.layer4.parameters(), "lr": 1e-5},
        {"params": model.fc.parameters(), "lr": 1e-4},
    ]
)

print("Fine-tuning layer4 and the classifier.")


## 6.3 CONTINUE TRAINING AND VALIDATE — Markdown + code: train further, evaluate each epoch, and retain the best validation checkpoint.

This continues fine-tuning, evaluates after each epoch using evaluate_on_validation, and saves a checkpoint whenever validation accuracy improves. Set best_val_acc to your best score so far—or 0.0 if this is your first validation checkpoint.

In [ ]:
from pathlib import Path
import torch

epochs = 5
best_val_acc = 0.0
best_checkpoint_path = Path("checkpoints/resnet18_best_finetuned.pth")
best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    total_examples = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        total_examples += batch_size

    train_loss = running_loss / total_examples
    val_loss, val_accuracy = evaluate_on_validation(
        model, val_loader, criterion, device
    )

    print(
        f"Epoch {epoch + 1}/{epochs} — "
        f"train loss: {train_loss:.4f} | "
        f"validation loss: {val_loss:.4f} | "
        f"validation accuracy: {val_accuracy:.4f}"
    )

    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        torch.save(
            {
                "epoch": epoch + 1,
                "val_accuracy": val_accuracy,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            },
            best_checkpoint_path,
        )
        print(f"Saved best checkpoint: {best_checkpoint_path}")


# 7.0. Tune the learning rate, batch-size, then GPU/CPU and data-handling settings using validation performance and training speed to compare trials Fine-tune selected pre-trained  

In [ ]:
## 7.1. IMPORTS 

## 7.2.0 TUNE LEARNING RATE — Markdown + code: compare learning-rate candidates while holding other settin fixed.  
Tune the `learning rate` first while keeping the model, batch size, data split, and training duration fixed. Try a small range of values, starting with the new classifier head; choose the rate that gives the best validation performance without unstable training. Start each trial from the same pretrained weights so results are comparable.

In [ ]:
# 7.2.0. TUNE THE LEARNING RATE 
learning_rates = [1e-5, 1e-4, 1e-3]
results = []

for lr in learning_rates:
    # Create a fresh model for each trial so trials start from the same weights.
    trial_model = models.resnet18(weights=weights)
    trial_model.fc = nn.Linear(trial_model.fc.in_features, num_classes)
    trial_model.to(device)

    # Freeze the pretrained backbone for this first round of tuning.
    for name, parameter in trial_model.named_parameters():
        parameter.requires_grad = name.startswith("fc.")

    trial_optimizer = optim.Adam(
        (p for p in trial_model.parameters() if p.requires_grad),
        lr=lr,
    )

    # Replace train_one_epoch and evaluate with your training/validation functions.
    for epoch in range(epochs):
        train_one_epoch(trial_model, train_loader, trial_optimizer, criterion, device)

    val_acc = evaluate_accuracy(trial_model, val_loader, device)
    results.append({"learning_rate": lr, "val_accuracy": val_acc})

print(results)


## 7.3. TUNE BATCH SIZE — Markdown + code: compare batch sizes using the selected learning rate.

Compare batch sizes while keeping the selected learning rate, data split, model setup, and number of epochs fixed. This example trains only the classifier head for each trial; if you’re tuning after fine-tuning, use the same unfreezing setup for every trial.

It assumes train_one_epoch and evaluate_on_validation are defined, and that train_set, val_set, weights, num_classes, best_learning_rate, criterion, and device are available.

In [ ]:
from torch.utils.data import DataLoader
from torchvision import models
from torch import nn, optim

batch_sizes = [16, 32, 64]
tuning_results = []

for batch_size in batch_sizes:
    # Use the same split, changing only the batch size.
    trial_train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True
    )
    trial_val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False
    )

    # Start each trial with the same pretrained backbone and a fresh head.
    torch.manual_seed(42)
    trial_model = models.resnet18(weights=weights)
    trial_model.fc = nn.Linear(trial_model.fc.in_features, num_classes)

    for parameter in trial_model.parameters():
        parameter.requires_grad = False
    for parameter in trial_model.fc.parameters():
        parameter.requires_grad = True

    trial_model = trial_model.to(device)
    trial_optimizer = optim.Adam(
        trial_model.fc.parameters(),
        lr=best_learning_rate,
    )

    for epoch in range(epochs):
        train_one_epoch(
            trial_model, trial_train_loader, trial_optimizer, criterion, device
        )

    val_loss, val_accuracy = evaluate_on_validation(
        trial_model, trial_val_loader, criterion, device
    )

    tuning_results.append({
        "stage": "batch_size",
        "batch_size": batch_size,
        "learning_rate": best_learning_rate,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
    })

for result in tuning_results:
    print(result)


After choosing a `learning rate`, compare a few `batch_size`values while keeping the `learning rate` and other settings fixed. \
\
Batch size affects memory use and training speed as well as optimization, so select based on validation performance and whether training fits in memory. Keep the train, validation, and test splits unchanged.


## 7.3 TUNE GPU/CPU AND DATA-LOADING SETTINGS — Markdown + code: compare device, `num_workers`, and `pin_memory`; record training time as well as validation accuracy.

### 7.3.1. IMPORTS

In [ ]:
#7.3.1.
import time

### 7.3.2 - TUNE FOR HARDWARE SETTINGS

This example compares data-loader settings on your available device. It keeps the model, batch size, and data split fixed, and records epoch time and validation accuracy for each num_workers setting.

In [ ]:
import time
import torch
from torch.utils.data import DataLoader

# Use the selected batch size from the previous tuning step.
worker_options = [0, 2, 4]
hardware_results = []

for num_workers in worker_options:
    trial_train_loader = DataLoader(
        train_set,
        batch_size=best_batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )
    trial_val_loader = DataLoader(
        val_set,
        batch_size=best_batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
    )

    # Start each trial from the same saved checkpoint for a fair comparison.
    trial_model = models.resnet18(weights=None)
    trial_model.fc = nn.Linear(trial_model.fc.in_features, num_classes)

    checkpoint = torch.load(
        "checkpoints/resnet18_adapted_initial.pth",
        map_location=device,
        weights_only=True,
    )
    trial_model.load_state_dict(checkpoint["model_state_dict"])
    trial_model.to(device)

    # Configure the same trainable layers and optimizer for each trial.
    for parameter in trial_model.parameters():
        parameter.requires_grad = False
    for parameter in trial_model.fc.parameters():
        parameter.requires_grad = True

    trial_optimizer = optim.Adam(
        trial_model.fc.parameters(),
        lr=best_learning_rate,
    )

    # Warm-up is excluded from timing to reduce first-iteration overhead.
    train_one_epoch(
        trial_model, trial_train_loader, trial_optimizer, criterion, device
    )
    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    train_one_epoch(
        trial_model, trial_train_loader, trial_optimizer, criterion, device
    )
    if device.type == "cuda":
        torch.cuda.synchronize()
    epoch_seconds = time.perf_counter() - start

    val_loss, val_accuracy = evaluate_on_validation(
        trial_model, trial_val_loader, criterion, device
    )

    hardware_results.append({
        "device": str(device),
        "num_workers": num_workers,
        "pin_memory": (device.type == "cuda"),
        "batch_size": best_batch_size,
        "epoch_seconds": epoch_seconds,
        "val_accuracy": val_accuracy,
    })

for result in hardware_results:
    print(result)


/# 7.1.3.2. \
`train_one_epoch`, `evaluate_accuracy`, `epochs`, `criterion`, and the selected values `best_learning_rate` and `best_batch_size` are placeholders for the functions and values established in the earlier notebook cells. \
\
After these staged experiments, retrain using the selected settings, choose the checkpoint with the best validation result, and evaluate it once on the held-out test set.

## 7.5 SUMMARIZE TUNING RESULTS — Markdown + code: compare trial settings and identify the chosen configuration.

For easy comparison, have each tuning trial append a row with the same fields to tuning_results. Use validation accuracy to select the configuration; use throughput to compare hardware settings.

In [ ]:
import pandas as pd

# Add one record per trial in Steps 7.1–7.3, for example:
# tuning_results.append({
#     "stage": "learning_rate",
#     "learning_rate": lr,
#     "batch_size": batch_size,
#     "device": str(device),
#     "num_workers": num_workers,
#     "val_accuracy": val_acc,
#     "images_per_second": images_per_second,
# })

tuning_df = pd.DataFrame(tuning_results)

# Review all trials, sorted by validation accuracy.
display(tuning_df.sort_values("val_accuracy", ascending=False))

# Select the highest-validation-accuracy trial.
# If there's a tie, prefer the faster trial when throughput is available.
sort_columns = ["val_accuracy"]
ascending_order = [False]

if "images_per_second" in tuning_df.columns:
    sort_columns.append("images_per_second")
    ascending_order.append(False)

best_trial = (
    tuning_df
    .sort_values(sort_columns, ascending=ascending_order)
    .iloc[0]
)

best_learning_rate = best_trial["learning_rate"]
best_batch_size = best_trial["batch_size"]

print("Selected configuration:")
print(best_trial.to_string())

# Save the comparison for your experiment records.
tuning_df.to_csv("tuning_results.csv", index=False)


# 8.0. Select the best checkpoint unising validation results, then evaluate it once on the held-out test set

Choose the checkpoint based on validation performance. Then load that checkpoint and evaluate it once on the held-out test set.

## 8.1. IMPORTS

In [ ]:
# 8.1. import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 8.2. LOAD THE BEST VALIDATION CHECKPOINT — Markdown + code: load the selected checkpoint into the model.

Run this cell once, after loading the checkpoint selected using validation results. It reports test loss, accuracy, per-class precision/recall/F1, and a confusion matrix.

In [ ]:
# 8.2.
# LOAD BEST CHECKPOINT AND TEST
checkpoint = torch.load(
    "best_finetuned_model.pth",
    map_location=device,
    weights_only=True,
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_labels, all_predictions = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        predictions = model(images).argmax(dim=1).cpu()
        all_predictions.extend(predictions.tolist())
        all_labels.extend(labels.tolist())

print("Test accuracy:", accuracy_score(all_labels, all_predictions))
print(classification_report(all_labels, all_predictions, target_names=dataset.classes))
print(confusion_matrix(all_labels, all_predictions))


## 8.3 EVALUATE ON THE HELD-OUT TEST SET — Markdown + code: report final test metrics once.

This cell records the model, selected training settings, and the test metrics from Step 8.2 in a small results table and saves it as a CSV. Adjust variable names to match those in your notebook.

In [ ]:
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Assumes model, test_loader, device, and dataset.classes are already defined.
criterion = torch.nn.CrossEntropyLoss()

model.to(device)
model.eval()

all_labels = []
all_predictions = []
total_loss = 0.0
total_examples = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        predictions = outputs.argmax(dim=1)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size

        all_labels.extend(labels.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())

test_loss = total_loss / total_examples
test_accuracy = accuracy_score(all_labels, all_predictions)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nPer-class metrics:")
print(classification_report(
    all_labels,
    all_predictions,
    labels=list(range(len(dataset.classes))),
    target_names=dataset.classes,
    zero_division=0,
))
print("Confusion matrix:")
print(confusion_matrix(
    all_labels,
    all_predictions,
    labels=list(range(len(dataset.classes))),
))


## 8.4 SUMMARIZE FINAL RESULTS — Markdown + code: record the model, selected settings, and test results.

This cell records the model, selected settings, class names, checkpoint path, and test metrics. It assumes these variables were set in earlier cells; adjust names if yours differ.

In [ ]:
from datetime import datetime, timezone
import pandas as pd

final_results = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "model": model_name,
    "pretrained_weights": str(weights),
    "classes": ", ".join(dataset.classes),
    "learning_rate": best_learning_rate,
    "batch_size": best_batch_size,
    "device": str(device),
    "num_workers": num_workers,
    "best_validation_accuracy": best_val_acc,
    "test_loss": test_loss,
    "test_accuracy": test_accuracy,
    "checkpoint": str(best_checkpoint_path),
}

results_df = pd.DataFrame([final_results])
display(results_df)

results_df.to_csv("final_test_results.csv", index=False)
